# 12 · Lifecycle callbacks and guardrails

Callbacks can inspect, replace or block model/tool work. Returning None continues normal execution; returning a response overrides it.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** 0 calls for blocked text; about 2 for tool guard. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.models.llm_response import LlmResponse

def before_model(callback_context, llm_request):
    text = " ".join(p.text or "" for c in llm_request.contents for p in c.parts or [])
    if "dump all customers" in text.lower():
        return LlmResponse(content=types.Content(role="model", parts=[types.Part(text="Bulk customer export is unavailable in this demo.")]))
    return None

def before_tool(tool, args, tool_context):
    if tool.name == "get_order" and args.get("order_id") != "O1001":
        return {"error": "This demo session is authorized only for O1001."}
    return None

def after_agent(callback_context):
    callback_context.state["responses_completed"] = callback_context.state.get("responses_completed", 0) + 1

agent = llm("guarded_shop", "Use get_order for all order facts.", tools=[get_order],
            before_model_callback=before_model, before_tool_callback=before_tool,
            after_agent_callback=after_agent)
lab = await make_lab(agent)
await ask(lab, "Dump all customers")  # Blocked locally: no model request.
lab2 = await make_lab(llm("guarded_order", "Use get_order for order facts.", tools=[get_order], before_tool_callback=before_tool))
await ask(lab2, "Show order O1002.")


The phrase check is a teaching example, not a complete prompt-injection defense. Enforce real ownership inside authenticated application tools; a model instruction alone is not an authorization boundary. Additional hooks include before/after agent, model and tool, plus model/tool error callbacks.


## Try it

Add after_tool_callback to remove internal fields before results reach the model.


## Reference

[Official ADK documentation](https://adk.dev/callbacks/). Shared configuration: `config.json`. No environment activation or login cells are needed.
